### Links and Resources
| Concept / Reference | Description |
|---------------------|-------------|
| [Delta Transaction Log](https://docs.databricks.com/en/delta/transaction-log.html) | Official overview of how the `_delta_log/` works |
| [Delta Protocol](https://github.com/delta-io/delta/blob/master/PROTOCOL.md) | Full open-source spec for every action type in the log |
| [DeltaTable.history()](https://docs.delta.io/latest/api/python/spark/index.html) | User-friendly view of the log — one row per commit |
| [DESCRIBE HISTORY (SQL)](https://docs.databricks.com/en/sql/language-manual/delta-describe-history.html) | SQL equivalent of `history()` |
| [Data Skipping](https://docs.databricks.com/en/delta/data-skipping.html) | How min/max stats stored in `add` actions speed up queries |
| [Checkpoints](https://docs.databricks.com/en/delta/transaction-log.html#checkpointing) | How Delta snapshots the log every 10 commits to speed up startup |

In [ ]:
import os, sys

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()
except ImportError:
    if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
        raise RuntimeError(
            f"Wrong Python kernel — 'databricks.connect' not found in:\n  {sys.executable}\n\n"
            "In VSCode: click the kernel selector (top-right of notebook) "
            "and choose the interpreter at:\n"
            r"  C:\Source\Dsa\DataBricks\.venv\Scripts\python.exe"
        )
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()


def display(df, n=20):
    df.show(n, truncate=False)

# The Delta Transaction Log

Every Delta table has a hidden `_delta_log/` directory sitting alongside its Parquet data files.  
This directory **is** the table — everything else (the Parquet files) is just payload.

```
my_table/
├── _delta_log/                         ← transaction log
│   ├── 00000000000000000000.json       ← commit 0 (CREATE)
│   ├── 00000000000000000001.json       ← commit 1 (INSERT)
│   ├── 00000000000000000002.json       ← commit 2 (DELETE)
│   ├── 00000000000000000003.json       ← commit 3 (OPTIMIZE)
│   ├── ...
│   ├── 00000000000000000010.json       ← commit 10
│   ├── 00000000000000000010.checkpoint.parquet  ← checkpoint (every 10th)
│   └── _last_checkpoint                ← points to the latest checkpoint
├── part-00000-abc.parquet              ← data file
├── part-00000-def.parquet              ← data file (older version)
└── ...
```

### How Delta reads a table

When you query a Delta table Spark does not simply read all Parquet files in the folder.  
It reads the transaction log first:

```
1. Find the latest checkpoint (from _last_checkpoint)
2. Replay all JSON commits after the checkpoint
3. Build the "active file list" = all add actions − all remove actions
4. Read only the Parquet files in the active file list
```

This is what makes ACID, time travel, and data skipping possible.

### Action types inside a commit JSON file

Each line in a `*.json` commit file is one **action** — a JSON object with a single top-level key:

| Action | Present in | Meaning |
|--------|-----------|----------|
| `commitInfo` | Every commit | Who did what, when, and with which parameters |
| `protocol` | First commit (or when upgraded) | Minimum reader/writer version and enabled features |
| `metaData` | First commit (or when schema/properties change) | Table schema, partition columns, table properties |
| `add` | Any commit that writes new files | A new Parquet file is now part of the table — includes path, size, and column stats |
| `remove` | Any commit that replaces or deletes files | A Parquet file is logically removed — still on disk until VACUUM |
| `txn` | Streaming writes | Idempotency token for exactly-once streaming |

## Setup — Create a Demo Table at a Known Volume Path

To read the `_delta_log/` files directly we need a table at a path we can address with Spark.  
A Unity Catalog Volume provides a stable, readable path that works from both local Databricks Connect and the cluster.

The table is written to a Volume path rather than registered in Unity Catalog — this means `DeltaTable.forPath` is used, and the `_delta_log/` is at a predictable location.

In [ ]:
from pyspark.sql import Row, functions as F
from delta.tables import DeltaTable
import json

# Volume path — predictable location for direct log inspection
DEMO_PATH = "/Volumes/population_metrics/landing/datasets/countries_dataset/countries_dataset/delta/txn_log_demo"
LOG_PATH  = f"{DEMO_PATH}/_delta_log"

rows = [
    Row(product_id=1,  name="Laptop Pro 15",      category="Electronics", price=1299.99, in_stock=True),
    Row(product_id=2,  name="Wireless Mouse",      category="Electronics", price=25.49,  in_stock=True),
    Row(product_id=3,  name="USB-C Hub",           category="Electronics", price=39.99,  in_stock=False),
    Row(product_id=4,  name="Standing Desk",       category="Furniture",   price=499.00, in_stock=True),
    Row(product_id=5,  name="Ergonomic Chair",     category="Furniture",   price=349.00, in_stock=True),
    Row(product_id=6,  name="Monitor Arm",         category="Furniture",   price=89.95,  in_stock=False),
]

# Write as Delta — this is commit 0
spark.createDataFrame(rows).write.format("delta").mode("overwrite").save(DEMO_PATH)

dt = DeltaTable.forPath(spark, DEMO_PATH)
print(f"Table written to: {DEMO_PATH}")
print(f"Log directory   : {LOG_PATH}")
display(dt.toDF().orderBy("product_id"))

# Part 1 — Listing the `_delta_log/` Directory

After the initial write there is exactly one file in `_delta_log/`: `00000000000000000000.json`.  
Each subsequent operation appends a new numbered file.

In [ ]:
# List the _delta_log directory using Spark's binaryFile reader (reads metadata only)
log_listing = (
    spark.read.format("binaryFile")
    .load(LOG_PATH)
    .select(
        F.regexp_extract("path", r"([^/]+)$", 1).alias("file"),
        F.col("length").alias("bytes"),
        F.col("modificationTime").alias("modified"),
    )
    .orderBy("file")
)

print("Files in _delta_log/ after commit 0:")
display(log_listing)

In [ ]:
# Add a few more commits so we have several log files to inspect
dt = DeltaTable.forPath(spark, DEMO_PATH)

# Commit 1 — INSERT two more rows
new_rows = spark.createDataFrame([
    Row(product_id=7, name="Notebook (A5)",  category="Stationery", price=4.99,  in_stock=True),
    Row(product_id=8, name="Coffee Mug",     category="Kitchen",    price=12.00, in_stock=True),
])
new_rows.write.format("delta").mode("append").save(DEMO_PATH)

# Commit 2 — UPDATE price
dt = DeltaTable.forPath(spark, DEMO_PATH)
dt.update("product_id = 1", {"price": "1099.99"})

# Commit 3 — DELETE out-of-stock rows
dt = DeltaTable.forPath(spark, DEMO_PATH)
dt.delete("in_stock = false")

# Commit 4 — OPTIMIZE
spark.sql(f"OPTIMIZE delta.`{DEMO_PATH}`")

print("Commits 1-4 applied.")
print("\n_delta_log/ contents after 5 commits:")
display(
    spark.read.format("binaryFile").load(LOG_PATH)
    .select(
        F.regexp_extract("path", r"([^/]+)$", 1).alias("file"),
        F.col("length").alias("bytes"),
    )
    .orderBy("file")
)

# Part 2 — Anatomy of a Commit JSON File

Each `*.json` commit file contains one JSON object per line — each line is one **action**.  
Spark can read all commit files at once using a wildcard path; the resulting DataFrame will have one column per action type, each being a struct (or `null` when that action is absent from a given line).

In [ ]:
# Read all JSON commit files as a single DataFrame
# Each row = one action line from one commit file
# Spark adds "_metadata.file_name" so we can see which commit a line came from
raw_log = (
    spark.read
    .option("recursiveFileLookup", "false")
    .json(f"{LOG_PATH}/0*.json")
    .withColumn("_source_file", F.input_file_name())
    .withColumn("_commit", F.regexp_extract("_source_file", r"(\d+)\.json$", 1).cast("long"))
)

print("Top-level columns in the raw log DataFrame:")
print([f.name for f in raw_log.schema.fields])
print(f"\nTotal action lines across all commits: {raw_log.count()}")

### `commitInfo` — The Who, What, and When of Each Commit

`commitInfo` is present in every commit. It maps directly to what you see in `history()` but is stored as raw JSON in the log.

In [ ]:
# Extract commitInfo rows — one per commit
commit_info = (
    raw_log
    .filter(F.col("commitInfo").isNotNull())
    .select(
        F.col("_commit").alias("version"),
        F.from_unixtime(F.col("commitInfo.timestamp") / 1000).alias("timestamp"),
        F.col("commitInfo.operation").alias("operation"),
        F.col("commitInfo.operationParameters").alias("parameters"),
        F.col("commitInfo.userName").alias("user"),
    )
    .orderBy("version")
)

print("commitInfo — one row per commit:")
display(commit_info)

### Comparing `commitInfo` to `history()`

`DeltaTable.history()` and `DESCRIBE HISTORY` are just a user-friendly projection over the same `commitInfo` entries you just read directly from the JSON files.

In [ ]:
dt = DeltaTable.forPath(spark, DEMO_PATH)

print("history() — the polished view of the same data:")
display(
    dt.history()
    .select("version", "timestamp", "operation", "operationParameters", "operationMetrics")
    .orderBy("version")
)

# SQL equivalent:
# spark.sql(f"DESCRIBE HISTORY delta.`{DEMO_PATH}`")

### `protocol` — Feature Contract Between Writer and Reader

The `protocol` action appears in the very first commit and whenever a new feature is enabled.  
It defines the minimum Delta reader/writer version required to understand this table.

| Field | Meaning |
|-------|---------|
| `minReaderVersion` | Minimum Delta reader version that can read this table |
| `minWriterVersion` | Minimum Delta writer version that can write to this table |
| `readerFeatures` | Named reader features enabled (e.g. `deletionVectors`) |
| `writerFeatures` | Named writer features enabled (e.g. `deletionVectors`, `invariants`) |

In [ ]:
protocol_row = (
    raw_log
    .filter(F.col("protocol").isNotNull())
    .select(
        F.col("_commit").alias("version"),
        F.col("protocol.minReaderVersion").alias("minReaderVersion"),
        F.col("protocol.minWriterVersion").alias("minWriterVersion"),
        F.col("protocol.readerFeatures").alias("readerFeatures"),
        F.col("protocol.writerFeatures").alias("writerFeatures"),
    )
)

print("protocol actions (appear when features change):")
display(protocol_row)

### `metaData` — Schema and Table Properties

`metaData` is written in the first commit and whenever the schema or table properties change.  
It stores the table's schema as a JSON string (`schemaString`) — the same string you see in `detail()`.

In [ ]:
metadata_row = (
    raw_log
    .filter(F.col("metaData").isNotNull())
    .select(
        F.col("_commit").alias("version"),
        F.col("metaData.id").alias("tableId"),
        F.col("metaData.createdTime").alias("createdTime"),
        F.col("metaData.partitionColumns").alias("partitionColumns"),
        F.col("metaData.configuration").alias("properties"),
        F.col("metaData.schemaString").alias("schemaString"),
    )
)

print("metaData actions (schema and properties):")
display(metadata_row)

# Pretty-print the schema JSON for readability
schema_json = metadata_row.select("schemaString").collect()[0]["schemaString"]
print("\nParsed schemaString:")
print(json.dumps(json.loads(schema_json), indent=2))

# Part 3 — `add` Actions: The Source of Truth for Active Files

An `add` action records that a new Parquet file has been written.  
It is the most important action in the log because it contains:

| Field | Meaning |
|-------|---------|
| `path` | Relative path to the Parquet file |
| `size` | File size in bytes |
| `modificationTime` | When the file was written |
| `dataChange` | `true` if the file contains new data rows (vs. OPTIMIZE-only compaction) |
| `stats` | **Column-level statistics** — `numRecords`, `minValues`, `maxValues`, `nullCount` per column |
| `partitionValues` | Partition column values (empty map if table is not partitioned) |
| `deletionVector` | Path to a deletion vector file, if rows in this file have been soft-deleted |

In [ ]:
# Extract all add actions across all commits
add_actions = (
    raw_log
    .filter(F.col("add").isNotNull())
    .select(
        F.col("_commit").alias("version"),
        F.regexp_extract(F.col("add.path"), r"([^/]+\.parquet)", 1).alias("file"),
        F.col("add.size").alias("bytes"),
        F.col("add.dataChange").alias("dataChange"),
        F.col("add.stats").alias("stats_json"),
    )
    .orderBy("version")
)

print("All add actions (one row = one Parquet file added to the table):")
display(add_actions)

### Column Statistics Inside `add` — The Engine of Data Skipping

The `stats` field in every `add` action is a JSON string containing per-column min, max, and null counts.  
Databricks reads these stats **without opening the Parquet file** — this is how it skips entire files for selective queries.

```
Query: WHERE product_id = 99

File A stats: minValues.product_id = 1, maxValues.product_id = 8
  → 99 is outside [1, 8] → SKIP this file

File B stats: minValues.product_id = 7, maxValues.product_id = 12
  → 99 is outside [7, 12] → SKIP this file

Result: 0 files read, 0 I/O cost
```

In [ ]:
# Parse the stats JSON from the first add action to inspect its structure
stats_raw = add_actions.filter(F.col("stats_json").isNotNull()).select("version", "file", "stats_json").collect()

for row in stats_raw:
    parsed = json.loads(row["stats_json"])
    print(f"\nCommit {row['version']} | {row['file']}")
    print(json.dumps(parsed, indent=2))

### What the Stats Tell Us

```json
{
  "numRecords": 6,
  "minValues": { "product_id": 1, "price": 89.95 },
  "maxValues": { "product_id": 6, "price": 1299.99 },
  "nullCount":  { "product_id": 0, "price": 0, ... }
}
```

- **`numRecords`** — how many rows are in this Parquet file (before any deletion vector is applied).
- **`minValues` / `maxValues`** — the range of each column's values. Delta uses these to prune files at query time without opening them.
- **`nullCount`** — how many nulls per column, used by the query optimiser for NOT NULL predicates.

Stats are collected for the first 32 columns by default, controlled by `delta.dataSkippingNumIndexedCols`.

# Part 4 — `remove` Actions: Logical Deletion of Files

A `remove` action records that a previously-added Parquet file is no longer part of the current table version.  
The file is **not deleted from storage** — only its reference is removed from the active file list.  
VACUUM removes the physical file later.

| Field | Meaning |
|-------|---------|
| `path` | Relative path to the now-removed Parquet file |
| `deletionTimestamp` | Unix ms timestamp when the remove action was written |
| `dataChange` | `true` if rows were actually deleted; `false` for OPTIMIZE (file replaced, not data changed) |

In [ ]:
# Extract all remove actions
remove_actions = (
    raw_log
    .filter(F.col("remove").isNotNull())
    .select(
        F.col("_commit").alias("version"),
        F.regexp_extract(F.col("remove.path"), r"([^/]+\.parquet)", 1).alias("file"),
        F.from_unixtime(F.col("remove.deletionTimestamp") / 1000).alias("logicallyRemovedAt"),
        F.col("remove.dataChange").alias("dataChange"),
    )
    .orderBy("version")
)

print("All remove actions (file is still on disk — just no longer in the active set):")
display(remove_actions)

# Notice: OPTIMIZE (commit 4) produces removes with dataChange=false
# because the rows were not changed — only reorganised into a new file

### `dataChange = false` on OPTIMIZE

When OPTIMIZE compacts files it writes new `add` entries (the compacted files) and `remove` entries (the old small files).  
Both have `dataChange = false` — no rows were actually added or removed; the data just moved into fewer files.  

Streaming readers use `dataChange` to distinguish real data changes from structural reorganisations, so they do not reprocess OPTIMIZE outputs.

# Part 5 — Reconstructing the Active File List

The set of Parquet files that constitute the **current version** of the table is computed as:

```
active files = { all add.path values } − { all remove.path values }
```

This is exactly what Delta Lake computes internally every time you query the table.  
Let's replicate that logic manually.

In [ ]:
# All files ever added (full path for joining)
all_added = (
    raw_log
    .filter(F.col("add").isNotNull())
    .select(F.col("add.path").alias("path"), F.col("_commit").alias("addedInVersion"))
)

# All files ever removed
all_removed = (
    raw_log
    .filter(F.col("remove").isNotNull())
    .select(F.col("remove.path").alias("path"))
)

# Active files = added MINUS removed
active_files = all_added.join(all_removed, on="path", how="left_anti")

print(f"Total files ever added  : {all_added.count()}")
print(f"Total files ever removed: {all_removed.count()}")
print(f"Currently active files  : {active_files.count()}")
print("\nActive Parquet files (what Delta reads for the current version):")
display(active_files.select(F.regexp_extract("path", r"([^/]+\.parquet)", 1).alias("file"), "addedInVersion").orderBy("addedInVersion"))

### Confirm Against `detail()`

`detail()` reports `numFiles` — this should match our manual active file count.

In [ ]:
dt = DeltaTable.forPath(spark, DEMO_PATH)
detail = dt.detail().select("numFiles", "sizeInBytes", "statistics").collect()[0]
print(f"detail() numFiles : {detail['numFiles']}   ← should match active_files.count() above")
print(f"detail() sizeInBytes: {detail['sizeInBytes']}")

# Part 6 — Time Travel: Re-building the Active File List for a Past Version

Time travel works by replaying only the log entries up to (and including) the requested version.  
For version `N`, Delta filters: keep only `add` and `remove` actions from commits `0 … N`.

```
Time travel to version 1 (after INSERT, before UPDATE):
  add actions from commits 0–1 : file_A, file_B
  remove actions from commits 0–1 : (none)
  active at v1 = {file_A, file_B}

Time travel to version 2 (after UPDATE):
  add actions 0–2    : file_A, file_B, file_C
  remove actions 0–2 : file_A  (UPDATE rewrote it as file_C)
  active at v2 = {file_B, file_C}
```

In [ ]:
def active_files_at_version(log_df, version: int):
    """Manually compute which Parquet files are active at a given version."""
    adds    = log_df.filter((F.col("add").isNotNull())    & (F.col("_commit") <= version)).select(F.col("add.path").alias("path"))
    removes = log_df.filter((F.col("remove").isNotNull()) & (F.col("_commit") <= version)).select(F.col("remove.path").alias("path"))
    return adds.join(removes, on="path", how="left_anti")

for v in range(5):
    count = active_files_at_version(raw_log, v).count()
    print(f"  Version {v}: {count} active file(s)")

In [ ]:
# Verify against actual time travel reads
for v in range(5):
    row_count = spark.read.format("delta").option("versionAsOf", v).load(DEMO_PATH).count()
    print(f"  VERSION AS OF {v}: {row_count} row(s)")

# Part 7 — Checkpoints

Reading the full log by replaying every JSON file from commit 0 becomes expensive on tables with thousands of commits.  
**Checkpoints** solve this: every 10th commit Delta writes a Parquet snapshot of the **entire active file list** at that point.

On startup, Delta:
1. Reads `_last_checkpoint` to find the latest checkpoint version.
2. Loads the checkpoint Parquet — this gives the full active file list at the checkpoint version in one read.
3. Replays only the JSON commits **after** the checkpoint.

```
Without checkpoint:  replay commits 0, 1, 2, ..., 47, 48, 49
With checkpoint v40: load checkpoint_40.parquet + replay commits 41–49  (9 files instead of 50)
```

The checkpoint interval is controlled by `delta.checkpointInterval` (default: 10).

In [ ]:
# Generate 10 commits to trigger automatic checkpointing
dt = DeltaTable.forPath(spark, DEMO_PATH)

for i in range(9, 15):     # commits 5 through 10 — commit 10 triggers the checkpoint
    dt.update(
        condition=f"product_id = {(i % 6) + 1}",
        set={"price": F.lit(float(i * 10))}
    )
    dt = DeltaTable.forPath(spark, DEMO_PATH)   # refresh handle
    print(f"  Commit {i - 4}: small UPDATE applied")

print("\n_delta_log/ after 10 commits:")
display(
    spark.read.format("binaryFile").load(LOG_PATH)
    .select(
        F.regexp_extract("path", r"([^/]+)$", 1).alias("file"),
        F.col("length").alias("bytes"),
    )
    .orderBy("file")
)
# Look for a .checkpoint.parquet file and _last_checkpoint

In [ ]:
# Read _last_checkpoint — it tells Delta which checkpoint is the latest
last_checkpoint_df = spark.read.json(f"{LOG_PATH}/_last_checkpoint")
print("_last_checkpoint contents:")
display(last_checkpoint_df)
# Fields: version (which checkpoint), size (number of actions in the checkpoint)

In [ ]:
# Read the checkpoint Parquet file — it is a snapshot of all active add/remove/metaData/protocol actions
checkpoint_version = last_checkpoint_df.collect()[0]["version"]
checkpoint_path    = f"{LOG_PATH}/{str(checkpoint_version).zfill(20)}.checkpoint.parquet"

checkpoint_df = spark.read.parquet(checkpoint_path)
print(f"Columns in checkpoint v{checkpoint_version}:")
print([f.name for f in checkpoint_df.schema.fields])
print(f"\nTotal actions in checkpoint: {checkpoint_df.count()}")

# Count by action type
action_counts = [
    (col, checkpoint_df.filter(F.col(col).isNotNull()).count())
    for col in ["add", "remove", "metaData", "protocol", "commitInfo", "txn"]
    if col in [f.name for f in checkpoint_df.schema.fields]
]
print("\nAction counts in checkpoint:")
for action, count in action_counts:
    if count > 0:
        print(f"  {action:12s}: {count}")

In [ ]:
# The add actions in the checkpoint ARE the active file list at the checkpoint version
# — this is the full table state captured in a single Parquet read
checkpoint_files = (
    checkpoint_df
    .filter(F.col("add").isNotNull())
    .select(
        F.regexp_extract(F.col("add.path"), r"([^/]+\.parquet)", 1).alias("file"),
        F.col("add.size").alias("bytes"),
        F.col("add.dataChange").alias("dataChange"),
    )
)
print(f"Active files captured in checkpoint v{checkpoint_version}:")
display(checkpoint_files)

### Why Checkpoints Matter for Performance

On a table with 10,000 commits and a checkpoint every 10 commits:

| Startup step | Without checkpoints | With checkpoints |
|-------------|--------------------|-----------------|
| Files to open | 10,000 JSON files | 1 Parquet + up to 9 JSON files |
| Data read | All 10,000 JSON log lines | Checkpoint Parquet + 9 JSON lines |
| Time | Seconds to minutes | Milliseconds |

The `delta.checkpointInterval` table property (default `10`) trades write overhead (writing the checkpoint) against read startup time.

# Part 8 — Log Retention

The transaction log itself has a separate retention policy from the data files.

| Property | Default | Controls |
|----------|---------|----------|
| `delta.logRetentionDuration` | `interval 30 days` | How long old JSON commit files are kept |
| `delta.deletedFileRetentionDuration` | `interval 7 days` | How long orphaned Parquet data files are kept (VACUUM) |

Log files older than `logRetentionDuration` are cleaned up automatically by `VACUUM` — it removes both orphaned data files **and** old JSON commit files beyond the retention window.

### Log retention vs time travel

Once a commit's JSON file is removed:
- The corresponding `VERSION AS OF n` query will fail.
- `history()` will no longer show that version.

The 30-day default is deliberately longer than the 7-day data file retention so that the audit trail outlives the time-travel window.

In [ ]:
# Inspect the current retention settings on our demo table
dt = DeltaTable.forPath(spark, DEMO_PATH)
props = dt.detail().select("properties").collect()[0]["properties"] or {}

log_retention  = props.get("delta.logRetentionDuration",         "interval 30 days (default)")
data_retention = props.get("delta.deletedFileRetentionDuration", "interval 7 days (default)")

print(f"delta.logRetentionDuration         : {log_retention}")
print(f"delta.deletedFileRetentionDuration : {data_retention}")

# Change log retention (example — 60 days for compliance)
# spark.sql(f"""
#     ALTER TABLE delta.`{DEMO_PATH}`
#     SET TBLPROPERTIES ('delta.logRetentionDuration' = 'interval 60 days')
# """)

# Summary

## `_delta_log/` File Types

| File | Created | Contains |
|------|---------|----------|
| `N.json` | Every commit | One JSON line per action: `commitInfo`, `add`, `remove`, `metaData`, `protocol` |
| `N.checkpoint.parquet` | Every 10th commit (configurable) | Snapshot of all active `add`/`metaData`/`protocol` actions at version N |
| `_last_checkpoint` | Updated on every checkpoint | JSON with `version` and `size` of the latest checkpoint |

## Action Types

| Action | Key fields | When present |
|--------|-----------|-------------|
| `commitInfo` | `operation`, `timestamp`, `userName`, `operationParameters`, `operationMetrics` | Every commit |
| `protocol` | `minReaderVersion`, `minWriterVersion`, `readerFeatures`, `writerFeatures` | First commit; on feature upgrade |
| `metaData` | `schemaString`, `partitionColumns`, `configuration` | First commit; on schema/property change |
| `add` | `path`, `size`, `stats` (min/max/nullCount), `dataChange`, `deletionVector` | Any commit writing new files |
| `remove` | `path`, `deletionTimestamp`, `dataChange` | Any commit replacing or deleting files |

## How Delta Reads a Table

```
Query issued
  └─► Read _last_checkpoint → find latest checkpoint version N
       └─► Load checkpoint_N.parquet → full active file list at version N
            └─► Replay JSON commits N+1 … latest
                 └─► Apply adds and removes → final active file list
                      └─► Apply data skipping (min/max stats from add.stats)
                           └─► Read only the surviving Parquet files
```

## Log vs Data Retention

| | Controls | Default | Cleaned by |
|-|---------|---------|------------|
| **Parquet data files** | `delta.deletedFileRetentionDuration` | 7 days | `VACUUM` |
| **JSON commit files** | `delta.logRetentionDuration` | 30 days | `VACUUM` |
| **Checkpoint files** | `delta.logRetentionDuration` | 30 days | `VACUUM` |